# 6 · Faceting and Polishing
*Data Visualization for Scientists & Public Health Professionals*

This final notebook turns a working plot into a finished one. **Faceting** breaks a busy chart into a grid of small, comparable panels. **Color** chosen well makes a figure clearer and accessible. **Themes** set the overall look in one line. And a short **makeover** ties the course together: the gap between a default chart and a presentable one is a handful of deliberate choices. Then the **capstone** returns to the rural-closures project one last time and carries it all the way to a presentation-ready figure.

### Learning objectives
- Split a plot into small multiples with faceting
- Tell figure-level seaborn functions from axes-level ones
- Choose a colormap that fits the data and is colorblind-safe
- Set a theme and a presentation context in one line
- Take a default chart and polish it for an audience

### Agenda
1. Small multiples with faceting
2. Color and accessibility
3. Themes and context
4. A makeover
5. **Capstone Part 2 — trend, relate, and present the closures**

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

Back to `diabetes_viz` for the teaching sections, with the age order fixed as before.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_viz.csv", low_memory=False)

age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
df.shape

## 1. Small multiples with faceting

When several groups overlap on one chart, a grid of **small multiples** — one panel per group, same axes — is often easier to read than overlaid distributions. seaborn's **figure-level** functions build these grids for you: pass a categorical column to `col=` (and optionally `row=`) and you get one panel per value. `displot` is the figure-level counterpart to `histplot`. Here, the distribution of length of stay, one panel per readmission outcome.

In [ ]:
g = sns.displot(data=df, x="time_in_hospital", col="readmitted",
                col_order=["NO", ">30", "<30"], binwidth=1, height=3.5)
g.set_axis_labels("Days in hospital", "Encounters")
g.set_titles("readmitted = {col_name}")
g.figure.suptitle("Length of stay by readmission outcome", y=1.03)
plt.show()

> **Note:** Figure-level vs. axes-level — the distinction that trips people up. Axes-level functions (`histplot`, `boxplot`, `scatterplot`, `lineplot`) draw onto an `Axes` you pass with `ax=`, and you compose figures yourself — everything you did in Notebooks 1–5. Figure-level functions (`displot`, `catplot`, `relplot`, `lmplot`) manage their **own** figure to lay out the grid, so they do **not** take `ax=`; instead they return a `FacetGrid` you customize through methods like `set_axis_labels` and `set_titles`. [Building multi-plot grids](https://seaborn.pydata.org/tutorial/axis_grids.html)

### Exercise 1 — A faceted comparison *(7 min)*

Use `sns.catplot` (the figure-level categorical plot) with `kind="box"` to draw **`num_medications` by `diabetesMed`**, one panel per **`readmitted`** group (`col="readmitted"`). Set the axis labels through the returned grid.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
g = sns.catplot(data=df, x="diabetesMed", y="num_medications",
                col="readmitted", col_order=["NO", ">30", "<30"],
                kind="box", height=3.8)
g.set_axis_labels("Diabetes med prescribed", "Number of medications")
g.set_titles("readmitted = {col_name}")
g.figure.suptitle("Medication count by treatment and readmission", y=1.03)
plt.show()
```

**Why this works.** `catplot` is the figure-level entry to the categorical family (`kind="box"`, `"violin"`, `"bar"`, `"count"`), so `col="readmitted"` lays out one boxplot panel per readmission group and returns a `FacetGrid`. You label it through the grid, not an `Axes`, because the grid owns the figure. The pattern — patients on diabetes meds carry more medications — holds across all three panels.

</details>

## 2. Color and accessibility

Color is data, not decoration, so match the palette to the data type: **qualitative** palettes (distinct hues) for unordered categories, **sequential** (light-to-dark of one hue) for ordered or continuous values, and **diverging** (two hues from a neutral middle) for values around a meaningful midpoint — the correlation heatmap used one. And make it accessible: a meaningful fraction of viewers cannot distinguish red from green, so seaborn's `"colorblind"` palette is a safe qualitative default.

In [ ]:
race_counts = df["race"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=race_counts.index, y=race_counts.values,
            hue=race_counts.index, palette="colorblind", legend=False, ax=ax)
ax.set_title("Encounters by race (colorblind palette)")
ax.set_xlabel("Race")
ax.set_ylabel("Encounters")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

> **Tip:** Two habits beyond palette choice: do not let color be the *only* thing distinguishing categories (position, order, and labels help too), and avoid encoding meaning in red-versus-green alone. [Choosing color palettes](https://seaborn.pydata.org/tutorial/color_palettes.html)

## 3. Themes and context

`sns.set_theme` sets the global look. **`style`** controls the background and gridlines (`"whitegrid"`, `"white"`, `"darkgrid"`, `"ticks"`); **`context`** scales every font and line for where the figure will live (`"paper"`, `"notebook"`, `"talk"`, `"poster"`). Set it **once** near the top of a notebook and every later plot inherits it. `"talk"` enlarges text for slides.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

stay_by_age = df.groupby("age")["time_in_hospital"].mean().reindex(age_order)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(age_order, stay_by_age.values, marker="o")
ax.set_title("Mean length of stay by age")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean days")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

> **Tip:** The fonts and lines are now sized for a projected slide, and because the setting is global, the remaining plots keep this look until you change it. To return to defaults, call `sns.set_theme()` with no arguments.

## 4. A makeover

Putting it together. Here is a chart with every default left untouched — no title, default axis names, arbitrary category order, one opaque color — followed by the deliberate-choice version.

In [ ]:
sns.set_theme()   # back to defaults, to show the starting point honestly

fig, ax = plt.subplots()
sns.barplot(data=df, x="race", y="num_lab_procedures", errorbar=None, ax=ax)
plt.show()

Now the polished version: a presentation context, an informative title and axis labels, bars ordered by value (race is unordered, so this is fair), a colorblind palette, and the figure saved for both slides and print.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

order = df.groupby("race")["num_lab_procedures"].mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df, x="race", y="num_lab_procedures", order=order,
            hue="race", hue_order=order, palette="colorblind", legend=False,
            errorbar=None, ax=ax)
ax.set_title("Mean lab procedures by race")
ax.set_xlabel("Race")
ax.set_ylabel("Mean lab procedures")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

fig.savefig("labs_by_race.png", dpi=150, bbox_inches="tight")   # slides/web
fig.savefig("labs_by_race.pdf", bbox_inches="tight")            # print, vector
plt.show()

> **Note:** Same data, same one bar per group — but the second chart can stand on its own in a deck or a paper. None of the changes were clever; they were the deliberate-choice versions of decisions matplotlib otherwise makes for you.

### Exercise 2 — Your own makeover *(10 min)*

Start from this bare chart of encounter counts per `age`, then polish it: keep the **natural age order** (age is ordered, so do *not* sort by value), add a title and axis labels, set the presentation context, and save it as a PNG at 150 dpi.

```python
sns.set_theme()
counts = df["age"].value_counts()
sns.barplot(x=counts.index, y=counts.values)
plt.show()
```

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
sns.set_theme(style="whitegrid", context="talk")

counts = df["age"].value_counts().reindex(age_order)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=counts.index, y=counts.values,
            hue=counts.index, hue_order=age_order, palette="colorblind",
            legend=False, ax=ax)
ax.set_title("Encounters by age band")
ax.set_xlabel("Age band")
ax.set_ylabel("Encounters")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig("encounters_by_age.png", dpi=150, bbox_inches="tight")
plt.show()
```

**Why this works.** `reindex(age_order)` fixes the order to the natural age sequence rather than by count — the right call for an ordered category, and the mirror of the makeover above where sorting by value *was* right. The `set_*` methods supply the labels, and `savefig` with `dpi` and `bbox_inches="tight"` writes a clean raster. Polishing is mostly habit.

</details>

## Wrap-up

You can now break a crowded chart into faceted small multiples, tell figure-level seaborn functions from axes-level ones, choose a palette that fits the data and stays accessible, set a theme and presentation context in one line, and take a default chart through to something you would put in front of an audience. Now use all of it — and everything from Session 2 — on the capstone.

---

## Capstone · Part 2 — Trend, relate, and present the closures

One last return to **rural hospital closures**. In Part 1 you counted and compared them; now you bring the Session 2 toolkit — trends over time, a relationship, small multiples, and presentation polish — and finish with a figure ready for a slide.

**Driving question.** Did rural closures accelerate over time, do the hardest-hit states carry the heaviest disease burden, and how does access hold up where hospitals remain?

We start, as before, from the cleaned closures data.

In [ ]:
closures = pd.read_csv(f"{BASE_URL}/rural_hospital_closures.csv")
closures = closures.drop(columns="Unnamed: 0")
closures["closure_year"] = closures["closure_year"].replace(1019, 2019)
closures["closure_type"] = (closures["closure_type"].str.strip().str.capitalize()
                            .replace({"Complet": "Complete", "Convertd": "Converted"}))
closures["beds"] = closures["beds"].fillna(closures["beds"].median())
q1, q3 = closures["beds"].quantile([0.25, 0.75]); iqr = q3 - q1
closures = closures[closures["beds"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
closures = closures.dropna(subset=["state"])
closures.shape

### Task 1 — The trend: did closures accelerate?

Count closures per `closure_year` and plot them as a line, fully labeled. In a comment, say whether closures sped up over the period.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
per_year = closures["closure_year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(per_year.index, per_year.values, marker="o")
ax.set_title("Rural hospital closures per year")
ax.set_xlabel("Year")
ax.set_ylabel("Closures")
fig.tight_layout()
plt.show()
# Yes: closures drift upward from a couple a year in the mid-2000s to high single
# digits in the 2010s-2020s, with a sharp spike around 2019.
```

**Why this works.** `value_counts().sort_index()` turns the year column into an ordered count series — a time trend — and a line reads acceleration far better than bars would. The line is the time-series skill from Notebook 5, on yearly stakes.

</details>

### Task 2 — The relationship: are the hardest-hit states the sickest?

To relate closures to health, we enrich the closures with county context. The cell below builds a state-level table — closures per state, and each state's population-weighted diabetes prevalence from the ACS/PLACES county data (`merge` and `groupby`, both from the pandas course), so you can focus on the chart.

In [ ]:
acs = pd.read_csv(f"{BASE_URL}/acs2017.csv")
places = pd.read_csv(f"{BASE_URL}/places.csv")
counties = acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="inner")

state_health = (counties.groupby("State")
                .apply(lambda g: np.average(g["hdiabetes"], weights=g["TotalPop"]),
                       include_groups=False)
                .rename("diabetes"))
per_state = closures["state"].value_counts().rename("closures")
state = pd.concat([per_state, state_health], axis=1, join="inner").reset_index(names="state")
state.head()

Now plot it. Draw a scatter with a linear trend line of **`diabetes` (x)** against **`closures` (y)**, fully labeled. In a comment, say whether states with a heavier diabetes burden tend to have lost more hospitals.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=state, x="diabetes", y="closures",
            line_kws={"color": "crimson"}, ax=ax)
ax.set_title("State diabetes burden vs. rural closures")
ax.set_xlabel("Diabetes prevalence (%, population-weighted)")
ax.set_ylabel("Rural hospital closures")
fig.tight_layout()
plt.show()
# Positive (about 0.44): states with higher diabetes prevalence tend to have lost more
# rural hospitals -- the places losing access are also the places that most need care.
```

**Why this works.** The enrichment turns two unrelated tables into one state-level frame, and `regplot` — the relationship tool from Notebook 4 — draws the association. The upward slope is the capstone's core finding: closures are not random; they concentrate where the chronic-disease burden is highest.

</details>

### Task 3 — Small multiples of remaining access

Where hospitals remain, how busy are the clinics? Load the daily `clinic_visits` and, using a **figure-level** function, draw the **weekly mean visits** as small multiples — one panel per `site`.

In [ ]:
visits = pd.read_csv(f"{BASE_URL}/clinic_visits.csv", parse_dates=["date"])
weekly = (visits.pivot_table(index="date", columns="site", values="visits")
          .resample("W").mean()
          .reset_index()
          .melt("date", var_name="site", value_name="visits"))
weekly.head()

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
g = sns.relplot(data=weekly, x="date", y="visits", col="site",
                kind="line", height=3.5, aspect=1.1)
g.set_axis_labels("Week", "Mean visits per day")
g.set_titles("{col_name}")
g.figure.suptitle("Weekly clinic visits by site", y=1.03)
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=45)
plt.show()
```

**Why this works.** `relplot` is the figure-level counterpart to `lineplot`, so `col="site"` lays out one time-series panel per clinic on shared axes — the faceting skill from this notebook, applied to the capstone. Side-by-side panels make the three sites honestly comparable, and all three share the same mid-year dip.

</details>

### Task 4 — The finale: one presentation-ready figure

Bring it home. For the **Hilltop** site, plot daily visits with a **7-day rolling mean** on top, **annotate the five-day March outage** with a shaded span and label, set a presentation theme, and save the figure as a PNG at 150 dpi. This is the chart you would put on a slide.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
sns.set_theme(style="whitegrid", context="talk")

hill = visits[visits["site"] == "Hilltop"].set_index("date")["visits"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hill.index, hill.values, linewidth=0.7, alpha=0.4, label="daily")
ax.plot(hill.index, hill.rolling(7).mean(), color="crimson", label="7-day mean")

start, end = pd.Timestamp("2025-03-10"), pd.Timestamp("2025-03-14")
ax.axvspan(start, end, color="crimson", alpha=0.15)
ax.annotate("5-day outage", xy=(start, ax.get_ylim()[1]),
            xytext=(8, -16), textcoords="offset points", color="crimson")

ax.set_title("Hilltop clinic: daily visits, 2025")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.legend()
fig.tight_layout()
fig.savefig("hilltop_2025.png", dpi=150, bbox_inches="tight")
plt.show()

sns.set_theme()   # reset so later work starts clean
```

**Why this works.** Every piece is a skill from Session 2: the rolling mean (Notebook 5), the annotated span (Notebook 5), and the theme, labels, and save (this notebook). Together they turn a raw daily series into a figure that explains itself — the outage marked, the trend legible, the text sized for a room.

</details>

### The insight — and one surprise

**Insight.** Rural closures are not a finished story from the past: they *accelerated*, climbing from a couple a year to high single digits with a spike around 2019. And they are not random — they concentrate in the states carrying the heaviest diabetes burden, so the communities losing hospitals are the ones that most need chronic-disease care.

**One surprise.** It is diabetes prevalence, more than income, that tracks closures: the state-level correlation with the diabetes burden (about 0.44) runs stronger than the one with poverty (about 0.30). The clearest signal for *where* rural access is disappearing is not simply how poor a state is, but how sick it is.

**That closes the course.** You started with the Figure/Axes model, worked through distributions, categories, relationships, and time series, learned to facet and polish — and carried one real project, rural hospital access, from a clean table all the way to a figure ready for an audience.